In [0]:
%pip install rasterio
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.9/36.9 MB 150.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd

presencias = pd.read_csv('/Volumes/especies_nativas/especies_nativas/bronze/Albizia_guachapele.csv')
print(f'Total de registros: {len(presencias)}')
print(presencias.head())
print(presencias[['Latitude','Longitude']].describe())

Total de registros: 156
              species  Latitude  Longitude
0  Albizia guachapele  6.552925 -75.827428
1  Albizia guachapele  6.569195 -75.854682
2  Albizia guachapele  6.558209 -75.831538
3  Albizia guachapele  6.553894 -75.830259
4  Albizia guachapele  6.547381 -75.827953
         Latitude   Longitude
count  156.000000  156.000000
mean     6.494892  -75.762641
std      0.040399    0.043323
min      6.421380  -75.854682
25%      6.453211  -75.826912
50%      6.501409  -75.738710
75%      6.531604  -75.732544
max      6.609583  -75.708083


In [0]:
import rasterio

with rasterio.open('/Volumes/especies_nativas/especies_nativas/bronze/bio9_HISTORICO.tif') as src:
    print('CRS:', src.crs)
    print('Bounds:', src.bounds)
    print('Resolución:', src.res)
    print('Tamaño (ancho x alto):', src.width, 'x', src.height)
    print('NoData value:', src.nodata)
    print('Número de bandas:', src.count)

CRS: EPSG:4326
Bounds: BoundingBox(left=-76.05, bottom=6.350000000000003, right=-75.63333333333333, top=6.700000000000003)
Resolución: (0.008333333333333333, 0.008333333333333333)
Tamaño (ancho x alto): 50 x 42
NoData value: -3.3999999521443642e+38
Número de bandas: 1


In [0]:
import rasterio
import numpy as np

with rasterio.open('/Volumes/especies_nativas/especies_nativas/bronze/bio9_HISTORICO.tif') as src:
    coords = list(zip(presencias['Longitude'], presencias['Latitude']))
    valores = [v[0] for v in src.sample(coords)]
    nodata = src.nodata

presencias['bio_9'] = valores
print(presencias[['Latitude', 'Longitude', 'bio_9']].head(10))
print()
print('Valores NoData encontrados:', (presencias['bio_9'] == nodata).sum())
print('Rango de bio_9 en presencias:', presencias['bio_9'].min(), 'a', presencias['bio_9'].max())

   Latitude  Longitude      bio_9
0  6.552925 -75.827428  25.600000
1  6.569195 -75.854682  23.816668
2  6.558209 -75.831538  25.600000
3  6.553894 -75.830259  25.600000
4  6.547381 -75.827953  25.683334
5  6.547471 -75.827936  25.683334
6  6.547743 -75.828080  25.683334
7  6.547840 -75.828078  25.683334
8  6.557888 -75.835422  25.316668
9  6.547900 -75.828079  25.683334

Valores NoData encontrados: 0
Rango de bio_9 en presencias: 22.566666 a 26.55


In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS especies_nativas.silver') 

DataFrame[]

In [0]:
import numpy as np
import rasterio

RASTER_PATH = '/Volumes/especies_nativas/especies_nativas/bronze/bio9_HISTORICO.tif'
BUFFER_DEG = 0.0083333  # ~1 pixel (~1 km) alrededor de cada presencia

with rasterio.open(RASTER_PATH) as src:
    arr = src.read(1)
    nodata = src.nodata
    transform = src.transform

    rows, cols = np.where(arr != nodata)
    xs, ys = rasterio.transform.xy(transform, rows, cols)
    xs = np.array(xs)
    ys = np.array(ys)
    vals = arr[rows, cols]

pres_lon = presencias['Longitude'].values
pres_lat = presencias['Latitude'].values

mask_keep = np.ones(len(xs), dtype=bool)
for plon, plat in zip(pres_lon, pres_lat):
    dist = np.sqrt((xs - plon) ** 2 + (ys - plat) ** 2)
    mask_keep &= (dist > BUFFER_DEG)

bg_lon = xs[mask_keep]
bg_lat = ys[mask_keep]
bg_bio9 = vals[mask_keep]

print(f'Pixeles totales validos: {len(xs)}')
print(f'Pixeles de background tras excluir buffer: {len(bg_lon)}')

Pixeles totales validos: 1266
Pixeles de background tras excluir buffer: 1208


In [0]:
import pandas as pd

background = pd.DataFrame({
    'Latitude': bg_lat,
    'Longitude': bg_lon,
    'bio_9': bg_bio9,
    'presencia': 0
})

presencias_final = presencias[['Latitude', 'Longitude', 'bio_9']].copy()
presencias_final['presencia'] = 1

dataset_silver = pd.concat([presencias_final, background], ignore_index=True)
print(dataset_silver['presencia'].value_counts())
print(dataset_silver.head())

presencia
0    1208
1     156
Name: count, dtype: int64
   Latitude  Longitude      bio_9  presencia
0  6.552925 -75.827428  25.600000          1
1  6.569195 -75.854682  23.816668          1
2  6.558209 -75.831538  25.600000          1
3  6.553894 -75.830259  25.600000          1
4  6.547381 -75.827953  25.683334          1


In [0]:
sdf = spark.createDataFrame(dataset_silver)
sdf.write.format('delta').mode('overwrite').saveAsTable('especies_nativas.silver.dataset_bio9')
print('Tabla guardada correctamente')

Tabla guardada correctamente


In [0]:
spark.sql('CREATE SCHEMA IF NOT EXISTS especies_nativas.gold')

DataFrame[]

In [0]:
from sklearn.model_selection import train_test_split

dataset_gold = dataset_silver.copy()

train_idx, test_idx = train_test_split(
    dataset_gold.index,
    test_size=0.2,
    stratify=dataset_gold['presencia'],
    random_state=42
)

dataset_gold['split'] = 'train'
dataset_gold.loc[test_idx, 'split'] = 'test'

print(dataset_gold.groupby(['split', 'presencia']).size())

split  presencia
test   0            242
       1             31
train  0            966
       1            125
dtype: int64


In [0]:
sdf_gold = spark.createDataFrame(dataset_gold)
sdf_gold.write.format('delta').mode('overwrite').saveAsTable('especies_nativas.gold.dataset_bio9_final')
print('Tabla Gold guardada correctamente')

Tabla Gold guardada correctamente
